<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/03_AI_Engineer/intermediate/05_function_calling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Function Calling & Tool Use with LLMs

> **Track:** AI Engineer | **Level:** Intermediate | **Time:** 2-3 hours

## Overview
Function calling (tool use) is the mechanism that transforms a passive LLM into an **active agent** capable of taking actions in the world. Instead of generating free-form text, the model can request to call predefined functions — search the web, query a database, run calculations, or interact with any API.

### What You'll Learn
- How function calling works under the hood
- Defining tool schemas with JSON Schema
- Implementing parallel tool calls
- Building an agent with 5+ tools
- Handling tool errors gracefully
- Anthropic tool use API

```bash
pip install openai anthropic
```

In [ ]:
# Install and import required libraries
import json
import math
import os
from typing import Any

import openai

client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY", "YOUR_API_KEY_HERE"))

print("OpenAI client ready.")
print(f"SDK version: {openai.__version__}")

## 1. What Is Function Calling?

When you send a message to an LLM with tools defined, the model can respond in two ways:
1. **Normal text reply** — the model answers directly
2. **Tool call request** — the model says "I need to call `get_weather(city='Paris')` to answer this"

Your code executes the function, sends the result back, and the model incorporates it into its final answer.

```
User → [LLM + Tools] → Tool Call Request
                           ↓
                     Your Code Executes Tool
                           ↓
               Tool Result → [LLM] → Final Answer
```

In [ ]:
# Define a simple tool and its schema

def get_current_weather(city: str, unit: str = "celsius") -> dict[str, Any]:
    """Stub: returns mock weather data for a city."""
    mock_data: dict[str, Any] = {
        "Paris": {"temp": 18, "condition": "Partly Cloudy", "humidity": 65},
        "London": {"temp": 12, "condition": "Rainy", "humidity": 80},
        "Tokyo": {"temp": 25, "condition": "Sunny", "humidity": 55},
        "New York": {"temp": 20, "condition": "Clear", "humidity": 50},
    }
    data = mock_data.get(city, {"temp": 22, "condition": "Unknown", "humidity": 60})
    if unit == "fahrenheit":
        data["temp"] = round(data["temp"] * 9/5 + 32)
    return {"city": city, "unit": unit, **data}

# JSON Schema definition for the tool
weather_tool: dict[str, Any] = {
    "type": "function",
    "function": {
        "name": "get_current_weather",
        "description": "Get the current weather in a given city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name, e.g. 'Paris'"},
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"], "default": "celsius"}
            },
            "required": ["city"]
        }
    }
}

print("Tool schema:")
print(json.dumps(weather_tool, indent=2))

## 2. Single Tool Call — Full Loop

The complete flow for a single tool call:
1. Send message + tool definitions to LLM
2. LLM returns a `tool_calls` response
3. Parse the call, execute locally
4. Send tool result back as a `tool` role message
5. LLM returns final answer

In [ ]:
# Implement the full single-tool call loop

def run_tool_call_loop(
    user_message: str,
    tools: list[dict],
    tool_registry: dict[str, callable]
) -> str:
    """Execute a complete tool call loop and return the final answer."""
    messages = [{"role": "user", "content": user_message}]

    # Step 1: Ask the model (it may request tool calls)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )

    assistant_msg = response.choices[0].message
    messages.append(assistant_msg)

    # Step 2: If tool calls were requested, execute them
    if assistant_msg.tool_calls:
        for tool_call in assistant_msg.tool_calls:
            fn_name = tool_call.function.name
            fn_args = json.loads(tool_call.function.arguments)

            print(f"  → Calling: {fn_name}({fn_args})")
            result = tool_registry[fn_name](**fn_args)

            # Step 3: Append tool result to messages
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result)
            })

        # Step 4: Get final answer
        final = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages
        )
        return final.choices[0].message.content

    return assistant_msg.content


# Test it
answer = run_tool_call_loop(
    "What's the weather like in Paris right now?",
    tools=[weather_tool],
    tool_registry={"get_current_weather": get_current_weather}
)
print("\nFinal answer:", answer)

## 3. Parallel Tool Calls & Multi-Tool Agent

Modern LLMs can request **multiple tool calls simultaneously**. We define 5 tools:
- `calculator` — math expressions
- `get_current_weather` — weather
- `search_web` — web search stub
- `query_database` — SQL stub
- `read_file` — file reader stub

In [ ]:
# Define 5 tools and a multi-tool agent

def calculator(expression: str) -> dict[str, Any]:
    """Safely evaluate a math expression."""
    try:
        result = eval(expression, {"__builtins__": {}}, vars(math))
        return {"result": result, "expression": expression}
    except Exception as e:
        return {"error": str(e)}

def search_web(query: str, num_results: int = 3) -> dict[str, Any]:
    """Stub: returns mock search results."""
    return {"query": query, "results": [
        {"title": f"Result {i+1} for '{query}'", "url": f"https://example.com/{i+1}"}
        for i in range(num_results)
    ]}

def query_database(sql: str) -> dict[str, Any]:
    """Stub: returns mock query results."""
    return {"sql": sql, "rows": [{"id": 1, "name": "Alice", "score": 95}], "row_count": 1}

def read_file(filepath: str) -> dict[str, Any]:
    """Stub: returns mock file contents."""
    return {"filepath": filepath, "content": f"Contents of {filepath} (mock)", "size_bytes": 1024}

# Tool registry
TOOL_REGISTRY: dict[str, callable] = {
    "get_current_weather": get_current_weather,
    "calculator": calculator,
    "search_web": search_web,
    "query_database": query_database,
    "read_file": read_file,
}

# Schema for all 5 tools
ALL_TOOLS = [
    weather_tool,
    {"type": "function", "function": {
        "name": "calculator",
        "description": "Evaluate a math expression. Use Python math functions like sqrt, sin, cos.",
        "parameters": {"type": "object", "properties": {
            "expression": {"type": "string", "description": "e.g. 'sqrt(144) + 2**8'"}
        }, "required": ["expression"]}
    }},
    {"type": "function", "function": {
        "name": "search_web",
        "description": "Search the web for recent information.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"},
            "num_results": {"type": "integer", "default": 3}
        }, "required": ["query"]}
    }},
    {"type": "function", "function": {
        "name": "query_database",
        "description": "Run a SQL query against the company database.",
        "parameters": {"type": "object", "properties": {
            "sql": {"type": "string", "description": "Valid SQL SELECT statement"}
        }, "required": ["sql"]}
    }},
    {"type": "function", "function": {
        "name": "read_file",
        "description": "Read the contents of a file by its path.",
        "parameters": {"type": "object", "properties": {
            "filepath": {"type": "string"}
        }, "required": ["filepath"]}
    }},
]

print(f"Registered {len(ALL_TOOLS)} tools: {[t['function']['name'] for t in ALL_TOOLS]}")

## 4. Anthropic Tool Use API

Anthropic's Claude also supports tool use with a slightly different API surface:
- Tools are defined with `input_schema` instead of `parameters`
- Tool results are sent as `user` role messages with `type: tool_result`
- Stop reason is `tool_use` instead of `tool_calls`

In [ ]:
# Anthropic tool use example
import anthropic

ant_client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY", "YOUR_KEY"))

# Anthropic tool schema format (uses input_schema instead of parameters)
claude_calculator_tool = {
    "name": "calculator",
    "description": "Evaluate a math expression using Python math functions.",
    "input_schema": {
        "type": "object",
        "properties": {
            "expression": {"type": "string", "description": "e.g. 'sqrt(144) + 2**8'"}
        },
        "required": ["expression"]
    }
}

def run_claude_tool_loop(user_message: str) -> str:
    """Execute a tool call loop using Claude's API."""
    messages = [{"role": "user", "content": user_message}]

    response = ant_client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        tools=[claude_calculator_tool],
        messages=messages
    )

    if response.stop_reason == "tool_use":
        tool_use_block = next(b for b in response.content if b.type == "tool_use")
        tool_result = calculator(tool_use_block.input["expression"])

        # Anthropic: tool result goes as user message
        messages.extend([
            {"role": "assistant", "content": response.content},
            {"role": "user", "content": [{
                "type": "tool_result",
                "tool_use_id": tool_use_block.id,
                "content": json.dumps(tool_result)
            }]}
        ])

        final = ant_client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            tools=[claude_calculator_tool],
            messages=messages
        )
        return final.content[0].text

    return response.content[0].text


print("Claude tool use function defined.")
print("Uncomment below to run (requires ANTHROPIC_API_KEY):")
print("# result = run_claude_tool_loop('What is sqrt(1764) + 15^2?')")
print("# print(result)")

## Exercises

1. **Add a `convert_currency` tool** that converts between USD, EUR, and GBP using mock exchange rates. Add it to `ALL_TOOLS` and test it with a query like "How much is 100 USD in EUR?"

2. **Implement error handling**: Modify `run_tool_call_loop` so that if a tool raises an exception, it returns a graceful error message to the LLM (e.g., `{"error": "Tool failed: <reason>"}`) rather than crashing the loop.

3. **Build a multi-turn agent**: Extend `run_tool_call_loop` to support **multiple rounds** of tool calls in a single conversation (the model calls tool A, gets the result, then calls tool B based on A's result). Test with: "Search for the current Python version, then calculate 3.12 * 1000."